In [ ]:
#| default_exp render

# render

> Base renderer, model capability flags, and the model registry.
>
> Each image generation model is registered here with its capabilities.
> The pipeline uses `get_renderer()` to instantiate the right backend.
> Individual model implementations live in `manhualizer/renderers/`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import asyncio
import importlib
from abc import ABC, abstractmethod
from pathlib import Path

from rich.console import Console

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from pydantic import BaseModel

_console = Console()

## Model Capabilities

In [ ]:
#| export
class ModelCapabilities(BaseModel):
    """Flags describing what an image generation model can do.

    The pipeline uses these flags to adapt its behaviour — for example,
    character sheet generation is only triggered when `reference_images` is True.
    """
    reference_images: bool = False
    """Accepts image inputs — enables character sheet workflow."""
    multi_image_input: bool = False
    """Can accept more than one reference image per call."""
    lora: bool = False
    """Supports LoRA weight loading."""
    negative_prompt: bool = False
    """Supports a negative prompt."""
    aspect_ratio_control: bool = True
    """Can control output image dimensions."""

## Model Registry

In [ ]:
#| export
class ModelSpec(BaseModel):
    """Static description of a registered image generation model."""
    name: str
    display_name: str
    renderer_class: str
    """Fully qualified class path, e.g. 'manhualizer.renderers.nanobanana.NanaBananaRenderer'."""
    capabilities: ModelCapabilities

In [ ]:
#| export
MODELS: dict[str, ModelSpec] = {
    "nanobanana": ModelSpec(
        name="nanobanana",
        display_name="NanaBanana (Google Imagen)",
        renderer_class="manhualizer.renderers.nanobanana.NanaBananaRenderer",
        capabilities=ModelCapabilities(
            reference_images=True,
            multi_image_input=True,
            negative_prompt=False,
            lora=False,
        ),
    ),
    "chatgpt-image": ModelSpec(
        name="chatgpt-image",
        display_name="ChatGPT Image Model (OpenAI)",
        renderer_class="manhualizer.renderers.chatgpt_image.ChatGPTImageRenderer",
        capabilities=ModelCapabilities(
            reference_images=True,
            multi_image_input=False,
            negative_prompt=False,
            lora=False,
        ),
    ),
    "seadreem": ModelSpec(
        name="seadreem",
        display_name="Seadreem",
        renderer_class="manhualizer.renderers.seadreem.SeadreemRenderer",
        capabilities=ModelCapabilities(
            reference_images=False,
            lora=True,
            negative_prompt=True,
        ),
    ),
    "flux-klein": ModelSpec(
        name="flux-klein",
        display_name="Flux Klein 9B",
        renderer_class="manhualizer.renderers.flux_klein.FluxKleinRenderer",
        capabilities=ModelCapabilities(
            reference_images=False,
            lora=True,
            negative_prompt=True,
        ),
    ),
    "comfyui": ModelSpec(
        name="comfyui",
        display_name="ComfyUI (local)",
        renderer_class="manhualizer.renderers.comfyui.ComfyUIRenderer",
        capabilities=ModelCapabilities(
            reference_images=True,
            multi_image_input=True,
            lora=True,
            negative_prompt=True,
        ),
    ),
}


def list_models() -> list[str]:
    """Return names of all registered models."""
    return list(MODELS.keys())

## Base Renderer

In [ ]:
#| export
class BaseRenderer(ABC):
    """Abstract base class for all image generation backends.

    Subclasses implement `render_async()` for a single panel. Batch rendering
    with concurrency control and resume support is handled here in
    `render_batch_async()`.
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        self.model_spec = model_spec
        self.config = config

    @abstractmethod
    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        """Render a single panel to an image file.

        Args:
            panel: The panel to render.
            output_dir: Directory to write the image into.
            output_cfg: Output format and dimensions.
            reference_images: Optional mapping of character name → character
                sheet image path. Only populated when the model's capabilities
                include `reference_images=True`.

        Returns:
            RenderResult with the path of the written image.
        """
        ...

    async def render_batch_async(
        self,
        panels: list[Panel],
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
        resume: bool = True,
        concurrency: int = 4,
    ) -> list[RenderResult]:
        """Render all panels concurrently up to `concurrency` at a time.

        If `resume` is True, panels whose output file already exists are
        skipped and a RenderResult pointing to the existing file is returned.

        Warn (don't fail) when LoRA config is set but the model doesn't
        support it.
        """
        if self.config.loras and not self.model_spec.capabilities.lora:
            _console.print(
                f"[yellow]render: LoRA config ignored — "
                f"{self.model_spec.display_name} does not support LoRA[/yellow]"
            )

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        sem = asyncio.Semaphore(concurrency)

        async def _render_one(panel: Panel) -> RenderResult:
            expected = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
            if resume and expected.exists():
                _console.print(f"[dim]render: skipping panel {panel.panel_number} (exists)[/dim]")
                return RenderResult(
                    panel_number=panel.panel_number,
                    image_path=expected,
                    backend_used=self.model_spec.name,
                    prompt_used=panel.visual_prompt,
                )
            async with sem:
                return await self.render_async(panel, output_dir, output_cfg, reference_images)

        tasks = [_render_one(p) for p in panels]
        results = await asyncio.gather(*tasks, return_exceptions=False)
        return list(results)

## Factory

In [ ]:
#| export
def get_renderer(model_name: str, config: RendererConfig) -> BaseRenderer:
    """Instantiate a renderer by model name.

    Looks up the model in the registry, imports its renderer class, and
    returns an initialised instance. Raises ValueError for unknown models.
    """
    if model_name not in MODELS:
        raise ValueError(
            f"Unknown model '{model_name}'. "
            f"Available: {list_models()}"
        )
    spec = MODELS[model_name]
    module_path, class_name = spec.renderer_class.rsplit(".", 1)
    module = importlib.import_module(module_path)
    cls = getattr(module, class_name)
    return cls(spec, config)

## Tests

In [ ]:
from manhualizer.render import MODELS, ModelCapabilities, list_models

# All expected models are registered
assert set(list_models()) == {"nanobanana", "chatgpt-image", "seadreem", "flux-klein", "comfyui"}

# Capability checks
assert MODELS["nanobanana"].capabilities.reference_images
assert MODELS["nanobanana"].capabilities.multi_image_input
assert not MODELS["nanobanana"].capabilities.lora

assert MODELS["chatgpt-image"].capabilities.reference_images
assert not MODELS["chatgpt-image"].capabilities.multi_image_input

assert MODELS["flux-klein"].capabilities.lora
assert MODELS["flux-klein"].capabilities.negative_prompt
assert not MODELS["flux-klein"].capabilities.reference_images

assert MODELS["seadreem"].capabilities.lora
assert not MODELS["seadreem"].capabilities.reference_images

assert MODELS["comfyui"].capabilities.lora
assert MODELS["comfyui"].capabilities.reference_images
assert MODELS["comfyui"].capabilities.multi_image_input
assert MODELS["comfyui"].capabilities.negative_prompt

print("Registry OK")
print("Models:", list_models())

In [ ]:
# Batch render with resume (using a stub renderer)
import asyncio, tempfile
from pathlib import Path
from manhualizer.render import BaseRenderer, MODELS, get_renderer
from manhualizer.models import Panel, RenderResult
from manhualizer.config import OutputConfig, RendererConfig

class StubRenderer(BaseRenderer):
    """Renderer that writes an empty file without calling any API."""
    async def render_async(self, panel, output_dir, output_cfg, reference_images=None):
        path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        path.write_bytes(b"stub")
        return RenderResult(
            panel_number=panel.panel_number,
            image_path=path,
            backend_used="stub",
            prompt_used=panel.visual_prompt,
        )

panels = [
    Panel(panel_number=i, scene_id="s1", location="X",
          action_description="does something", visual_prompt=f"prompt {i}")
    for i in range(1, 4)
]
output_cfg = OutputConfig()
spec = MODELS["flux-klein"]
cfg = RendererConfig()
renderer = StubRenderer(spec, cfg)

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp)
    results = asyncio.run(renderer.render_batch_async(panels, out, output_cfg, resume=False, concurrency=2))
    assert len(results) == 3
    assert all(r.image_path.exists() for r in results)

    # Resume: pre-existing files are skipped
    results2 = asyncio.run(renderer.render_batch_async(panels, out, output_cfg, resume=True, concurrency=2))
    assert len(results2) == 3

print("Batch render + resume OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()